In [ ]:
import matplotlib.pylab as plt
import matplotlib as mpl
import xarray as xr
import pint_xarray
import numpy as np
import cftime
from functools import partial

from pism_terra.processing import integrate_rate, preprocess_netcdf, normalize_timeseries

ref_year = "1990"

In [ ]:
ds_free = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_1985_free//output/scalar/basin_g900m_id_CESM2-WACCM_historical_free_1985-01-01_2015-01-01.nc")
ds_free = ds_free.expand_dims({"uq_id": ["free"]})
ds_long = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_long_prescribed/output/scalar/basin_g3600m_id_CESM2-WACCM_historical_prescribed_1985-01-01_2015-01-01.nc")
ds_long = ds_long.expand_dims({"uq_id": ["presribed_long"]})
# ds_prescribed = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_1985_prescribed/output/scalar/basin_g900m_id_CESM2-WACCM_historical_free_1985-01-01_2015-01-01.nc")
# ds_prescribed = ds_prescribed.expand_dims({"uq_id": ["prescribed"]})
# ds_inv = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_2007_inv_func/output/scalar/basin_g1500m_id_CESM2-WACCM_uq_1_historical_free_2007-01-01_2015-01-01.nc")
# ds_inv = ds_inv.expand_dims({"uq_id": ["inv"]})
# ds_s = ds_s.expand_dims({"uq_id": ["all"]}).convert_calendar("standard", use_cftime=False).resample(time='MS').mean('time')
#ds = xr.merge([ds_free, ds_prescribed, ds_inv], compat="no_conflicts", join="outer")
ds = xr.merge([ds_free, ds_long], compat="no_conflicts", join="outer")
ds = ds.sel(basin="GIS")

In [ ]:
ds = ds.convert_calendar("standard", use_cftime=False).resample(time='MS').mean('time').pint.quantify()

In [ ]:
grace = xr.open_dataset("/Users/andy/base/pism-ragis/data/grace/greenland_mass_balance.nc").squeeze().pint.quantify()
mankoff = xr.open_dataset("/Users/andy/base/pism-ragis/data/mass_balance/mankoff_greenland_mass_balance_clean.nc").pint.quantify()
mankoff = mankoff.sum(dim="region").resample(time='MS').mean("time").pint.to("Gt/yr")

sigma = 2.0
mankoff_cmb = integrate_rate(mankoff.MB)
mankoff_cmb = (mankoff_cmb - mankoff_cmb.sel(time=ref_year, method="nearest"))
mankoff_mb = mankoff.MB.resample(time='YS').mean("time")
mankoff_mb_err = mankoff.MB_err.resample(time='YS').mean("time")
mankoff_smb = mankoff.SMB.resample(time='YS').mean("time")
mankoff_smb_err = mankoff.SMB_err.resample(time='YS').mean("time")
mankoff_glf = -mankoff.D.resample(time='YS').mean("time")
mankoff_glf_err = -mankoff.D_err.resample(time='YS').mean("time")


In [ ]:
mass = ds.ice_mass_glacierized
mass_from_d = integrate_rate(ds.tendency_of_ice_mass_due_to_surface_mass_flux) + integrate_rate(ds.tendency_of_ice_mass_due_to_discharge)
d = ds.grounding_line_flux_nonneg * xr.DataArray(900).pint.quantify("m") ** 2
mass_from_glf = integrate_rate(ds.tendency_of_ice_mass_due_to_surface_mass_flux) + integrate_rate(d)

mass = mass - mass.sel(time=ref_year, method="nearest")
mass = mass.pint.to("Gt")
mass_from_d = mass_from_d - mass_from_d.sel(time=ref_year, method="nearest")
mass_from_d = mass_from_d.pint.to("Gt")
mass_from_glf = mass_from_glf - mass_from_glf.sel(time=ref_year, method="nearest")
mass_from_glf = mass_from_glf.pint.to("Gt")
#mass = (ds.tendency_of_ice_mass.pint.to("Gt/yr") - xr.DataArray(400).pint.quantify("Gt/yr")).cumsum(dim="time") 
#mass = mass - mass.sel(time="2002", method="nearest")

fig, ax = plt.subplots(1, 1)
mass.plot(hue="uq_id", ax=ax, lw=1, add_legend=True)
ax.set_prop_cycle(None)
mass_from_d.plot(hue="uq_id", ax=ax, lw=1, ls="dotted", add_legend=True)
ax.set_prop_cycle(None)
mass_from_glf.plot(hue="uq_id", ax=ax, lw=1, ls="dashed", add_legend=True)
ax.set_prop_cycle(None)
#grace.cumulative_mass_balance.plot(ax=ax, color="#DC267F")
mankoff_cmb.plot(ax=ax, lw=2, color="0.5")
ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))


In [ ]:
fig, ax = plt.subplots(1, 1)
mass.plot(hue="uq_id", ax=ax, lw=1, add_legend=True)
ax.set_prop_cycle(None)
mankoff_cmb.plot(ax=ax, lw=2, color="0.5")
ax.set_xlim(np.datetime64("1985"), np.datetime64("2000"))

In [ ]:
glf = ds.tendency_of_ice_mass_due_to_discharge
#glf = d.pint.to("Gt/yr")
smb = ds.tendency_of_ice_mass_due_to_surface_mass_flux
mb = smb + glf 

rc_params = {
    "font.size": 6,
        # Add other rcParams settings if needed
}

with mpl.rc_context(rc=rc_params):

    fig, axs = plt.subplots(3, 1, sharex=True, figsize=(4.8, 4.4))
    axs[0].fill_between(mankoff_mb.time, mankoff_mb - sigma * mankoff_mb_err, mankoff_mb + sigma * mankoff_mb_err, color="0.5", alpha=0.25, lw=0)
    axs[1].fill_between(mankoff_mb.time, mankoff_smb - sigma * mankoff_smb_err, mankoff_smb + sigma * mankoff_smb_err, color="0.5", alpha=0.25, lw=0)
    axs[2].fill_between(mankoff_mb.time, mankoff_glf - sigma * mankoff_glf_err, mankoff_glf + sigma * mankoff_glf_err, color="0.5", alpha=0.25, lw=0)
    
    mankoff_mb.resample(time='YS').mean('time').plot(ax=axs[0], color="0.5", lw=2)
    mankoff_smb.resample(time='YS').mean('time').plot(ax=axs[1], color="0.5", lw=2)
    mankoff_glf.resample(time='YS').mean('time').plot(ax=axs[2], color="0.5", lw=2)
    for k, (da, ls) in enumerate([(mb, "solid"), (smb, "dotted"), (glf, "dashed")]):
        ax = axs[k]
        # for u in da["uq_id"].values:
        #     #da.sel(uq=u).plot(ax=ax, color=palette[u], ls=ls)
        #     da.sel(uq_id=u).plot(ax=ax, color=palette[u], ls="solid", lw=2)
        da.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1.5, add_legend=False if k > 0 else True)        
        ax.set_title(None)
        ax.set_xlabel(None)
        ax.axhline(0, color="k", lw=0.5, ls="dotted")
        ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))
        #ax.set_ylim(-1000, 1000)
    axs[-1].set_xlabel("Year")
    fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6.4, 6.2))
ax.fill_between(mankoff_mb.time, mankoff_mb - sigma * mankoff_mb_err, mankoff_mb + sigma * mankoff_mb_err, color="0.5", alpha=0.25, lw=0)
ax.fill_between(mankoff_mb.time, mankoff_smb - sigma * mankoff_smb_err, mankoff_smb + sigma * mankoff_smb_err, color="0.5", alpha=0.25, lw=0)
ax.fill_between(mankoff_mb.time, mankoff_glf - sigma * mankoff_glf_err, mankoff_glf + sigma * mankoff_glf_err, color="0.5", alpha=0.25, lw=0)

mankoff_mb.resample(time='YS').mean('time').plot(ax=ax, color="0.5", lw=2, ls="solid")
mankoff_smb.resample(time='YS').mean('time').plot(ax=ax, color="0.5", lw=2, ls="dashed")
mankoff_glf.resample(time='YS').mean('time').plot(ax=ax, color="0.5", lw=2, ls="dotted")

mb.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1.5, ls="solid", add_legend=True)        
ax.set_prop_cycle(None)
smb.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1.5, ls="dashed", add_legend=False)        
ax.set_prop_cycle(None)
glf.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1.5, ls="dotted", add_legend=False)     
ax.set_prop_cycle(None)


ax.set_title(None)
ax.axhline(0, color="k", lw=0.5, ls="dotted")
ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))
ax.set_ylim(-800, 800)

In [ ]:
dss = []
_ds = ds.tendency_of_ice_mass_due_to_discharge.expand_dims({"method": ["ice_discharge"]})
dss.append(_ds)
_ds = (ds.grounding_line_flux * xr.DataArray(900).pint.quantify("m") ** 2).expand_dims({"method": ["grounding_line_flux"]})
dss.append(_ds)
_ds = (ds.grounding_line_flux_nonneg * xr.DataArray(900).pint.quantify("m") ** 2).expand_dims({"method": ["neg_grounding_line_flux"]})
dss.append(_ds)
pism_glf = xr.concat(dss, dim="method") 

pism_cmb = integrate_rate(pism_glf) + integrate_rate(ds.tendency_of_ice_mass_due_to_surface_mass_flux)

fig, ax = plt.subplots(1, 1, figsize=(6.4, 6.2))
ax.fill_between(mankoff_mb.time, mankoff_glf - sigma * mankoff_glf_err, mankoff_glf + sigma * mankoff_glf_err, color="0.5", alpha=0.25, lw=0)

mankoff_glf.resample(time='YS').mean('time').plot(ax=ax, color="0.5", lw=2, ls="solid")
pism_glf.resample(time='YS').mean('time').plot(hue="method", ax=ax, lw=1, ls="solid")
ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))

fig, ax = plt.subplots(1, 1, figsize=(6.4, 6.2))
mankoff_cmb.plot(ax=ax, lw=2, color="0.5")

pism_cmb.resample(time='YS').mean('time').plot(hue="method", ax=ax, lw=1, ls="solid")
ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))

In [ ]:
from pathlib import Path
p = list(Path("/Users/andy/base/pism-terra/2026_08_ismip7_hist_2007_inv_penalty/output/inverse/").glob("inv_g1500m_id_CESM2-WACCM_uq_*_2006-01-01_2007-01-01.nc"))
dss = []
for f in p:
    _ds = xr.open_dataset(f, chunks="auto")
    w = _ds.pism_config.attrs["inverse.tikhonov.penalty_weight"]
    dss.append(_ds.expand_dims(penalty_weight=[w]))
inv_ds = xr.concat(dss, dim="penalty_weight")

In [ ]:
fig, ax = plt.subplots(1, 1)
inv_ds.inv_J_misfit.plot(hue="penalty_weight", ax=ax)
ax.set_yscale("log")

In [ ]:
res = (inv_ds.inv_residual / inv_ds.velbar_mag)
res.plot(col="penalty_weight", col_wrap=3, vmax=10)

In [ ]:
inv_ds.inv_residual.plot(col="penalty_weight", col_wrap=3, vmax=100)

In [ ]:
from pathlib import Path
p = list(Path("/Users/andy/base/pism-terra/2026_08_ismip7_hist_2007_inv_penalty/output/scalar/").glob("basin_g1500m_id_CESM2-WACCM_uq_*_historical_free_2007-01-01_2015-01-01.nc"))
dss = []
for f in p:
    _ds = xr.open_dataset(f, chunks="auto")
    w = _ds.pism_config.attrs["inverse.tikhonov.penalty_weight"]
    dss.append(_ds.expand_dims(penalty_weight=[w]))
    print(w)
inv_scalar_ds = xr.concat(dss, dim="penalty_weight")

In [ ]:
inv_scalar_ds.sel(basin="GIS").ice_mass_glacierized.plot(hue="penalty_weight")

In [ ]:
inv_scalar_ds.ice_mass_glacierized

In [ ]:
pism_glf

In [ ]:
kitp = xr.open_dataset("/Users/andy/base/pism-terra/2026_06_kitp_median_v4/output/scalar/basin_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
ismip7 = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_1985_free/output/scalar/basin_g900m_id_CESM2-WACCM_historical_free_1985-01-01_2015-01-01.nc")

In [ ]:
kitp_smb = kitp.isel(time=slice(0, 10)).tendency_of_ice_mass_due_to_surface_mass_flux.mean(dim="time")
ismip7_smb = ismip7.sel(time=slice("1985", "1994")).tendency_of_ice_mass_due_to_surface_mass_flux.mean(dim="time")

In [ ]:
kitp_smb

In [ ]:
ismip7_smb

In [ ]:
mankoff_smb.sel(time=slice("1990", "2019")).mean('time')

In [ ]:
kitp = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_kitp_v4/output/basins/scalar_GIS_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc").pint.quantify()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6.4, 2.2))
((kitp.ice_mass-kitp.isel(time=0).ice_mass).pint.to("Gt") / xr.DataArray(362.5).pint.quantify("Gt/mm")).plot(ax=ax)
((kitp.ice_mass_glacierized-kitp.isel(time=0).ice_mass_glacierized).pint.to("Gt") / xr.DataArray(362.5).pint.quantify("Gt/mm")).plot(ax=ax)

In [ ]:
cumulative_vars = ["ice_mass", "ice_mass_glacierized"]

In [ ]:
kitp_old_scalar = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_kitp_v4/output/scalar/scalar_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc").drop_vars("pism_config").pint.quantify()
kitp_old_scalar = normalize_timeseries(kitp_old_scalar.resample(time="YS").mean(), variables=cumulative_vars, reference_date=cftime.DatetimeNoLeap(1, 1, 1)).squeeze()

kitp_old_clipped = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_kitp_v4/output/basins/scalar_GIS_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc").drop_vars("pism_config").pint.quantify()
kitp_old_clipped = normalize_timeseries(kitp_old_clipped.resample(time="YS").mean(), variables=cumulative_vars, reference_date=cftime.DatetimeNoLeap(1, 1, 1)).squeeze()

kitp_old_clipped["grounding_line_flux"] = kitp_old_clipped["grounding_line_flux"].pint.to("Gt m^-2 yr^-1") * xr.DataArray(1200).pint.quantify("m") ** 2
kitp_old_clipped["grounding_line_flux_nonneg"] = kitp_old_clipped["grounding_line_flux_nonneg"].pint.to("Gt m^-2 yr^-1") * xr.DataArray(1200).pint.quantify("m") ** 2

In [ ]:
kitp_new_scalar = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_kitp_debm_v4/output/scalar/scalar_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0311-01-01.nc").drop_vars("pism_config").pint.quantify()
kitp_new_clipped = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_kitp_debm_v4/output/processed_scalar/gis_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0311-01-01.nc").drop_vars("pism_config").pint.quantify()

kitp_new_scalar = normalize_timeseries(kitp_new_scalar.resample(time="YS").mean(), variables=cumulative_vars, reference_date=cftime.DatetimeNoLeap(1, 1, 1)).squeeze()
kitp_new_clipped = normalize_timeseries(kitp_new_clipped.resample(time="YS").mean(), variables=cumulative_vars, reference_date=cftime.DatetimeNoLeap(1, 1, 1)).squeeze().drop_vars(["glacier_id", "glacier_id_name"])

kitp_new_clipped["grounding_line_flux"] = kitp_new_clipped["grounding_line_flux"].pint.to("Gt m^-2 yr^-1") * xr.DataArray(1200).pint.quantify("m") ** 2
kitp_new_clipped["grounding_line_flux_nonneg"] = kitp_new_clipped["grounding_line_flux_nonneg"].pint.to("Gt m^-2 yr^-1") * xr.DataArray(1200).pint.quantify("m") ** 2


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6.4, 2.2))
(kitp_old_clipped.ice_mass.pint.to("Gt") / xr.DataArray(-362.5).pint.quantify("Gt/mm")).plot(ax=ax)
(kitp_old_scalar.ice_mass.pint.to("Gt") / xr.DataArray(-362.5).pint.quantify("Gt/mm")).plot(ax=ax)
(kitp_new_scalar.ice_mass.pint.to("Gt") / xr.DataArray(-362.5).pint.quantify("Gt/mm")).plot(ax=ax)
(kitp_new_clipped.ice_mass.pint.to("Gt") / xr.DataArray(-362.5).pint.quantify("Gt/mm")).plot(ax=ax)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6.4, 2.2))
(kitp_old_clipped.tendency_of_ice_mass_due_to_surface_mass_flux.pint.to("Gt/yr")).plot(ax=ax)
(kitp_old_scalar.tendency_of_ice_mass_due_to_surface_mass_flux.pint.to("Gt/yr")).plot(ax=ax)
(kitp_new_scalar.tendency_of_ice_mass_due_to_surface_mass_flux.pint.to("Gt/yr")).plot(ax=ax)


In [ ]:
_dss = []
_dss.append(kitp_old_scalar.expand_dims(exp_id= ["old_scalar"]))
_dss.append(kitp_old_clipped.expand_dims(exp_id= ["old_clipped"]))
_dss.append(kitp_new_scalar.expand_dims(exp_id = ["new_scalar"]))
_dss.append(kitp_new_clipped.expand_dims(exp_id = ["new_clipped"]))
kitp = xr.concat(_dss, dim="exp_id", join="outer")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6.4, 4.2))
# (kitp.tendency_of_ice_mass_due_to_surface_mass_flux.pint.to("Gt/yr")).plot(hue="exp_id", ax=ax)
# ax.set_prop_cycle(None)
# (kitp.tendency_of_ice_mass_due_to_discharge.pint.to("Gt/yr")).plot(hue="exp_id", ax=ax)
# ax.set_prop_cycle(None)
(kitp.grounding_line_flux.pint.to("Gt/yr")).plot(hue="exp_id", ax=ax)
ax.set_prop_cycle(None)
(kitp.grounding_line_flux_nonneg.pint.to("Gt/yr")).plot(hue="exp_id", ax=ax)
ax.set_ylim(-500, 0)

In [ ]:
normalize_cumulative_variables

In [ ]:
kitp_new_clipped

In [ ]:
new = kitp_new_scalar.sel({"time": slice(cftime.DatetimeNoLeap(11, 1, 1), cftime.DatetimeNoLeap(311,1,1))})
new["time"] = [cftime.DatetimeNoLeap(y, 1, 1) for y in range(1, len(new.time) + 1)]

In [ ]:
new.ice_mass.to_dataframe().reset_index()

In [ ]:
import pandas as pd
def adjust_timeseries(ds: xr.Dataset, variables: list) -> pd.DataFrame:

    ds = ds.drop_vars(["time_bounds"], errors="ignore").drop_dims("nv", errors="ignore")
    ds = ds.sel({"time": slice(cftime.DatetimeNoLeap(11, 1, 1), cftime.DatetimeNoLeap(311,1,1))}).resample(time="YS").mean()
    ds["time"] = [cftime.DatetimeNoLeap(y, 1, 1) for y in range(1, len(ds.time) + 1)]
    ds = normalize_timeseries(ds, variables, cftime.DatetimeNoLeap(1,1,1))
    return ds

variables = ["ice_mass", "ice_mass_glacierized"]

kitp_ds = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_kitp_debm_v4/output/scalar/scalar_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0311-01-01.nc").drop_vars("pism_config")

kitp_adj_ds = adjust_timeseries(kitp_ds, variables)
kitp_adj_ds.to_netcdf("kitp.nc")
df = kitp_adj_ds.to_dataframe().reset_index()
df.to_csv("kitp.csv")

In [ ]:
for v in q.data_vars:
    print(q[v].pint.units)

In [ ]:
u = q[v].pint.units

In [ ]:
!ls

In [ ]:
import seaborn as sns

In [ ]:
sns.catplot?